# Feature Engineering — Advisor Risk Scoring

Reads the three Advisor 360 source tables, engineers features, creates a synthetic
`at_risk` label, and writes the flat training table `advisor_risk_features`.

**Features engineered:**
- **Book profile**: advisor-vs-team and advisor-vs-region ratios per metric, % metrics below benchmark
- **Tool adoption**: total sessions, tools adopted count, session ratio vs team
- **Holdings**: total AUM, fund count, max fund %, HHI concentration index

**Label**: `at_risk` = 1 when advisor underperforms benchmarks, has low tool adoption, and concentrated portfolio

In [0]:
from pyspark.sql import functions as F

from src.config import CFG
from src.utils.logger import get_logger

logger = get_logger(__name__)

# CFG applies the job's catalog/schema parameters (prod job → prod catalog);
# interactively it falls back to config.yaml.
fq = CFG.fq

# Use the latest available month across all three tables
LATEST_MONTH = (
    spark.table(fq("book_profile"))
    .select(F.max("Month_End_Date__c").alias("latest"))
    .collect()[0]["latest"]
)

logger.info(f"Catalog/Schema : {CFG.catalog}.{CFG.schema}")
logger.info(f"Latest month   : {LATEST_MONTH}")
logger.info(f"Training table : {fq('training')}")

In [0]:
# ── book_profile_360 → advisor-level benchmark ratios ─────────────
bp_df = (
    spark.table(fq("book_profile"))
    .filter(F.col("Month_End_Date__c") == LATEST_MONTH)
)

bp_features = (
    bp_df
    .withColumn(
        "vs_team",
        F.when(F.col("Team_Avg__c") > 0,
               F.col("Advisor_Value__c") / F.col("Team_Avg__c"))
        .otherwise(0.0),
    )
    .withColumn(
        "vs_region",
        F.when(F.col("Region_Avg__c") > 0,
               F.col("Advisor_Value__c") / F.col("Region_Avg__c"))
        .otherwise(0.0),
    )
    .withColumn(
        "below_team",
        (F.col("Advisor_Value__c") < F.col("Team_Avg__c")).cast("int"),
    )
    .withColumn(
        "below_region",
        (F.col("Advisor_Value__c") < F.col("Region_Avg__c")).cast("int"),
    )
    .groupBy("Salesforce_Id__c")
    .agg(
        F.round(F.avg("Advisor_Value__c"), 2).alias("avg_advisor_value"),
        F.round(F.avg("vs_team"), 2).alias("avg_vs_team_ratio"),
        F.round(F.avg("vs_region"), 2).alias("avg_vs_region_ratio"),
        F.sum("below_team").alias("metrics_below_team"),
        F.sum("below_region").alias("metrics_below_region"),
        F.count("*").alias("_num_metrics"),
    )
    .withColumn("pct_below_team", F.round(F.col("metrics_below_team") / F.col("_num_metrics"), 2))
    .withColumn("pct_below_region", F.round(F.col("metrics_below_region") / F.col("_num_metrics"), 2))
    .drop("_num_metrics")
)

logger.info(f"bp_features: {bp_features.count()} advisors, {len(bp_features.columns)} cols")

In [0]:
# ── tools_adoption_360 → session aggregates ─────────────────────
ta_df = (
    spark.table(fq("tools_adoption"))
    .filter(F.col("Month_End_Date__c") == LATEST_MONTH)
)

ta_features = (
    ta_df
    .withColumn("is_adopted", (F.col("Advisor_Sessions__c") > 0).cast("int"))
    .withColumn(
        "vs_team_sessions",
        F.when(F.col("Team_Sessions__c") > 0,
               F.col("Advisor_Sessions__c") / F.col("Team_Sessions__c"))
        .otherwise(0.0),
    )
    .groupBy("Salesforce_Id__c")
    .agg(
        F.round(F.sum("Advisor_Sessions__c"), 2).alias("total_advisor_sessions"),
        F.sum("is_adopted").alias("tools_adopted_count"),
        F.round(F.avg("vs_team_sessions"), 2).alias("avg_sessions_vs_team"),
    )
)

logger.info(f"ta_features: {ta_features.count()} advisors, {len(ta_features.columns)} cols")

In [0]:
# ── advisor_holdings_360 → portfolio concentration metrics ───────
ah_df = (
    spark.table(fq("advisor_holdings"))
    .filter(F.col("Month_End_Date__c") == LATEST_MONTH)
)

ah_features = (
    ah_df
    .withColumn("pct_sq", (F.col("Percentage__c") / 100) ** 2)
    .groupBy("Salesforce_Id__c")
    .agg(
        F.round(F.sum("Value__c"), 2).alias("total_aum"),
        F.count("*").alias("num_funds"),
        F.round(F.max("Percentage__c"), 2).alias("max_fund_pct"),
        F.round(F.sum("pct_sq"), 2).alias("hhi_concentration"),
    )
)

logger.info(f"ah_features: {ah_features.count()} advisors, {len(ah_features.columns)} cols")

In [0]:
# ── Join all feature groups ─────────────────────────────────────────
advisor_features = (
    bp_features
    .join(ta_features, on="Salesforce_Id__c", how="inner")
    .join(ah_features, on="Salesforce_Id__c", how="inner")
)

# ── Synthetic label: at_risk ───────────────────────────────────────
# An advisor is "at risk" when they:
#   1. Underperform vs team on >50% of book profile metrics
#   2. Have adopted fewer than 4 of 6 available tools
#   3. Have a concentrated portfolio (HHI > 0.25)
advisor_features = advisor_features.withColumn(
    "at_risk",
    (
        (F.col("pct_below_team") > 0.5)
        & (F.col("tools_adopted_count") < 4)
        & (F.col("hhi_concentration") > 0.25)
    ).cast("int"),
)

# ── Write to training table ────────────────────────────────────────
advisor_features.write.format("delta").mode("overwrite").saveAsTable(fq("training"))

total = advisor_features.count()
at_risk_count = advisor_features.filter(F.col("at_risk") == 1).count()
logger.info(
    f"{fq('training')} written: {total} rows, "
    f"{len(advisor_features.columns) - 2} features (excl. Salesforce_Id__c + at_risk), "
    f"at_risk=1: {at_risk_count} ({at_risk_count / total * 100:.1f}%)"
)

display(advisor_features.limit(10))

# ── Write serving features table (no label, for endpoint lookup) ───
serving_df = advisor_features.drop("at_risk").dropDuplicates(["Salesforce_Id__c"])
serving_df.write.format("delta").mode("overwrite").saveAsTable(fq("serving_features"))
logger.info(
    f"{fq('serving_features')} written: {serving_df.count()} rows, "
    f"{len(serving_df.columns)} columns (features + Salesforce_Id__c, no label)"
)